# 02 — Sentiment Classification with an LSTM (TensorFlow/Keras)

This notebook trains an LSTM-based classifier on the IMDB movie review dataset (built into Keras) to predict whether a review is positive or negative.

**What you'll see:**
- How to preprocess variable-length text sequences (padding/truncating)
- How an embedding layer feeds into an LSTM
- Many-to-one RNN usage: a whole sequence → a single output label

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

tf.random.set_seed(42)

## 1. Load and preprocess the IMDB dataset

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f"Training examples: {len(x_train)}")
print(f"Test examples: {len(x_test)}")
print(f"Example review (as word indices): {x_train[0][:20]}")

In [ ]:
# Pad/truncate all sequences to the same length so they can be batched
x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)

print(x_train.shape, x_test.shape)

## 2. Build the LSTM model

In [ ]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation="sigmoid")  # binary classification: positive/negative
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 3. Train

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=3,
    batch_size=64
)

## 4. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

## 5. Try a custom review

In [ ]:
word_index = imdb.get_word_index()

def encode_review(text, max_len=MAX_LEN):
    tokens = text.lower().split()
    encoded = [word_index.get(w, 0) + 3 for w in tokens]  # offset matches Keras IMDB encoding
    encoded = [i if i < VOCAB_SIZE else 0 for i in encoded]
    return pad_sequences([encoded], maxlen=max_len)

sample = "this movie was absolutely wonderful and touching"
encoded_sample = encode_review(sample)
prediction = model.predict(encoded_sample, verbose=0)[0][0]

print(f"Review: {sample}")
print(f"Predicted sentiment score: {prediction:.4f} ({'Positive' if prediction > 0.5 else 'Negative'})")

## Try it yourself

- Swap `LSTM(64)` for `GRU(64)` or `SimpleRNN(64)` and compare training speed/accuracy.
- Add a second stacked LSTM layer with `return_sequences=True` on the first layer.
- Try a **Bidirectional** wrapper: `Bidirectional(LSTM(64))` — does accuracy improve?
- Increase `epochs` and watch for overfitting (validation loss increasing while training loss decreases).